In [0]:
dbutils.widgets.text("environment", "dev")
dbutils.widgets.text('processing_date', '2024-02-20', 'Inclusive processing date')

processing_date = dbutils.widgets.get('processing_date')
env = dbutils.widgets.get("environment")
catalog = f'{env}_banking'
ods_schema = 'ods'
dwh_schema='gold'
print(env, processing_date)

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
def full_table_name(schema, name):
    return f'{catalog}.{schema}.{name}'


In [0]:
run_to = F.to_timestamp(F.lit(f"{processing_date} 23:59:59"))
# watermarks = spark.table(table(dwh_schema, "dwh_watermark")).select(
#     "target_table",
#     F.coalesce("watermark_to", F.to_timestamp(F.lit("1900-01-01"))).alias("watermark_from"),
# )

def incremental(ods_name, target_name):
    source = spark.table(full_table_name(ods_schema, ods_name)).alias("s")
    # wm = watermarks.filter(F.col("target_table") == target_name).select("watermark_from").first()
    lower = '1900-01-01' #wm["watermark_from"] if wm else spark.sql("select timestamp('1900-01-01')").first()[0]
    return source.filter(
        (F.col("s.processing_date") > F.lit(lower)) & (F.col("s.processing_date") <= run_to)
    )

def merge_df(df, target_name, keys):
    target = DeltaTable.forName(spark, full_table_name(dwh_schema, target_name))
    condition = " AND ".join([f"t.{k}=s.{k}" for k in keys])
    assign = {c: f"s.{c}" for c in df.columns}
    target.alias("t").merge(df.alias("s"), condition).whenMatchedUpdate(
        set=assign
    ).whenNotMatchedInsert(values=assign).execute()

def hash_key(*cols):
    return F.sha2(F.concat_ws("|", *[F.coalesce(F.col(c).cast("string"), F.lit("")) for c in cols]),256,)


## Date Dimension

In [0]:
dates = spark.range(1).select(F.explode(F.sequence(
            F.to_date(F.lit("2000-01-01")),
            F.to_date(F.lit("2035-12-31")),
            F.expr("interval 1 day"),
        )
    ).alias("CalendarDate")
)

date_df = dates.select(
    F.date_format("CalendarDate", "yyyyMMdd").cast("int").alias("DateSK"),
    "CalendarDate",
    F.dayofmonth("CalendarDate").alias("DayOfMonth"),
    F.month("CalendarDate").alias("MonthNumber"),
    F.date_format("CalendarDate", "MMMM").alias("MonthName"),
    F.quarter("CalendarDate").alias("QuarterNumber"),
    F.year("CalendarDate").alias("YearNumber"),
    (F.last_day("CalendarDate") == F.col("CalendarDate")).alias("IsMonthEnd"),
).dropDuplicates(["DateSK"])

date_df.write.mode("overwrite").saveAsTable(full_table_name(dwh_schema,"dim_date"))

In [0]:
date_df.show()

## Type-1 dimensions

In [0]:
spark.sql(f"SELECT * FROM {catalog}.ods.branch").show()

In [0]:
branch = (
    incremental("branch", "dim_branch")
    .select("branch_id", "branch_name", "city", "state", "region", "processing_date")
    .withColumn("BranchSK", hash_key("branch_id"))
    .selectExpr( "BranchSK", "branch_id BranchID", "branch_name BranchName", "city City", "state State",
                 "region Region", "processing_date SourceIngestedAt",)
    )
merge_df(branch, "dim_branch", ["BranchID"])

In [0]:
product = (
    incremental("product", "dim_product")
    .withColumn("ProductSK", hash_key("product_id"))
    .selectExpr( "ProductSK", "product_id ProductID", "product_name ProductName", "product_category ProductCategory",
                 "annual_fee AnnualFee", "processing_date SourceIngestedAt")
    )
merge_df(product, "dim_product", ["ProductID"])

## SCD-2 Dimensions rebuilds

In [0]:
def rebuild_account_scd():
    """Maintain account history from a current-state ODS snapshot.

    ods_account has one current row per account. Existing dimension rows are
    therefore the history source: a changed current snapshot expires the
    current row and appends a new SCD-2 version.
    """
    changed = (
        incremental("account", "dim_account")
        .select( "account_id", "primary_customer_id", "branch_id", "account_type", "account_status", "open_date", "close_date", "currency_code", "source_modified_at", "processing_date",
        )
        .dropDuplicates(["account_id"]) # Read the incremental records
    )
    if changed.limit(1).count() == 0:
        print("No new changes are found")
        return

    current = (
        spark.table(full_table_name(dwh_schema, "dim_account")).filter(F.col("IsCurrent")) # fetch the latest/current account info
        .select("AccountID", "PrimaryCustomerID", "BranchID", "AccountType", "AccountStatus", "OpenDate", "CloseDate", "CurrencyCode", "StartDate", "AccountSK",
        )
    )

    effective = F.coalesce(F.to_date("source_modified_at"), F.to_date("open_date"),F.to_date(F.lit(processing_date)),)

    source = (changed.withColumn("StartDate", effective)
        .withColumn("AccountSK", hash_key("account_id", "StartDate"))
        .selectExpr( "AccountSK", "account_id AccountID", "primary_customer_id PrimaryCustomerID", "branch_id BranchID", "account_type AccountType", "account_status AccountStatus", "to_date(open_date) OpenDate", "to_date(close_date) CloseDate", "currency_code CurrencyCode", "StartDate", "to_date('9999-12-31') EndDate", "true IsCurrent", "processing_date SourceIngestedAt",
        )
    )

    tracked_columns = ["PrimaryCustomerID", "BranchID", "AccountType", "AccountStatus", "OpenDate", "CloseDate", "CurrencyCode"]

    # Combine new incoming account information from ODS (source) -> with existing latest IsCurrent=true account information in DWH (current)
    # On initial load current will be empty as there will be no account information in DWH, only new data from ODS (source) will be inserted in DWH
    joined = source.alias("s").join(current.alias("c"), F.col("s.AccountID") == F.col("c.AccountID"), "left")   
    unchanged = F.lit(True)
    # Weill keep only records in df with below 2 conition
    # 1. c.AccountSK IS NULL i.e we have new accounts i.e. current state of DWH doen't have the AccountSK
    filters = []
    for column in tracked_columns:
        filters.append(f's.{column} <=> c.{column}')

    to_insert = joined.filter(f"c.AccountSK IS NULL OR NOT ({' AND '.join(filters)})").select("s.*")

    # SCD Type 2: Expire the existing current record for any AccountID that has changed.
    # For each AccountID in to_insert, find the currently active record in the DWH (t.IsCurrent = true) and expire it by:
    # - Setting EndDate to the StartDate of the new incoming record.
    # - Setting IsCurrent to false.
    # This closes the validity period of the existing version immediately before the new version becomes effective.
    DeltaTable.forName(spark, full_table_name(dwh_schema, "dim_account")).alias("t").merge(
        to_insert.select("AccountID", "StartDate").alias("s"),
        condition="t.AccountID = s.AccountID AND t.IsCurrent = true",
    ).whenMatchedUpdate(set={"EndDate": "s.StartDate", "IsCurrent": "false"}).execute()

    # SCD Type 2: Insert the new version of the account record.
    # The incoming record contains the date its StartDate, it is appended as a new row, becoming the current version of the account.
    to_insert.write.mode("append").saveAsTable(full_table_name(dwh_schema,"dim_account"))

rebuild_account_scd()

In [0]:
spark.read.table(full_table_name(dwh_schema,"dim_account")).show()

In [0]:
def rebuild_customer_scd():
    # Fetches all customers from ODS (customer_history) that have changed since the last load.
    changed = incremental("customer_history", "dim_customer").select("customer_id").distinct()
    if changed.limit(1).count() == 0:
        return
    # Fetches all previous historical versions of the customers from ODS (customer_history) that have updated since the last load.
    customers_all_versions = (spark.table(full_table_name(ods_schema, "customer_history"))
        .join(changed, "customer_id")
    )
    w2 = Window.partitionBy("customer_id").orderBy(F.to_date("effective_from_date"))
    result = (
        customers_all_versions.withColumn("StartDate", F.to_date("effective_from_date"))
        .withColumn("EndDate", F.coalesce(F.lead(F.to_date("effective_from_date")).over(w2), F.to_date(F.lit("9999-12-31"))))
        .withColumn("IsCurrent", F.lead("effective_from_date").over(w2).isNull())
        .withColumn("CustomerSK", hash_key("customer_id", "StartDate"))
        .selectExpr( 
            "CustomerSK", "customer_id CustomerID", "full_name FullName", "address_line_1 AddressLine1", "city City", "state State", 
            "postal_code PostalCode", "customer_segment CustomerSegment", "kyc_status KYCStatus", "StartDate", "EndDate", "IsCurrent", "processing_date SourceIngestedAt"
        )
    )
    result.write.mode("append").saveAsTable(full_table_name(dwh_schema, "dim_customer"))

rebuild_customer_scd()


In [0]:
spark.read.table(full_table_name(dwh_schema,"dim_customer")).show()

In [0]:
def rebuild_ownership_bridge():
    changed = incremental("account_customer_history", "bridge_account_customer").select("account_id", "customer_id").distinct()
    if changed.limit(1).count() == 0:
        return
    source = spark.table(full_table_name(ods_schema, "account_customer_history")).join(changed, ["account_id", "customer_id"])
    
    w = Window.partitionBy("account_id", "customer_id").orderBy(F.to_date("effective_from_date"))
    bridge = (
        source.withColumn("StartDate", F.to_date("effective_from_date"))
        .withColumn("EndDate", F.coalesce(F.lead(F.to_date("effective_from_date")).over(w),F.to_date(F.lit("9999-12-31"))))
        .withColumn("IsCurrent", F.lead("effective_from_date").over(w).isNull())
        .withColumn("BridgeSK", hash_key("account_id", "customer_id", "StartDate"))
        .withColumn("CustomerSK", hash_key("customer_id", "StartDate"))
        .selectExpr( 
            "BridgeSK", "account_id AccountID", "customer_id CustomerID", "CustomerSK", "ownership_role OwnershipRole", "ownership_percent OwnershipPercent", 
            "StartDate", "EndDate", "IsCurrent", "processing_date SourceIngestedAt",
        )
    )
    bridge.write.mode("append").saveAsTable(full_table_name(dwh_schema, "bridge_account_customer"))

rebuild_ownership_bridge()

In [0]:
spark.read.table(full_table_name(dwh_schema,"bridge_account_customer")).show()

## Incremental facts and bridges

In [0]:
accounts = spark.table(full_table_name(dwh_schema, "dim_account")).select(
    "AccountID", "AccountSK", "PrimaryCustomerID", "BranchID", "StartDate", "EndDate"
)
branches = spark.table(full_table_name(dwh_schema, "dim_branch")).select("BranchID", "BranchSK")
customers = spark.table(full_table_name(dwh_schema, "dim_customer")).select("CustomerID", "CustomerSK", "StartDate", "EndDate")

In [0]:
tx_source = incremental("transaction", "fact_transactions").alias("t")
tx = (
    tx_source.join(
        accounts.alias("a"),
        (F.col("t.account_id") == F.col("a.AccountID"))
        & (F.to_date("t.transaction_date") >= F.col("a.StartDate"))
        & (F.to_date("t.transaction_date") < F.col("a.EndDate")),
    )
    .join(branches.alias("b"), F.col("a.BranchID") == F.col("b.BranchID"))
    .join(customers.alias("c"),
        (F.col("a.PrimaryCustomerID") == F.col("c.CustomerID"))
        & (F.to_date("t.transaction_date") >= F.col("c.StartDate"))
        & (F.to_date("t.transaction_date") < F.col("c.EndDate")),
    )
    .selectExpr("CAST(date_format(t.transaction_date,'yyyyMMdd') AS INT) DateSK", "t.transaction_id TxnID", "t.account_id AccountID", "a.AccountSK", "c.CustomerSK", "b.BranchSK", "t.transaction_timestamp TransactionTimestamp", "t.transaction_type TransactionType", "t.transaction_amount TransactionAmount", "t.currency_code CurrencyCode", "t.merchant_name MerchantName", "t.merchant_category MerchantCategory", "t.transaction_status TransactionStatus", "t.processing_date SourceIngestedAt",
    )
)
merge_df(tx, "fact_transactions", ["TxnID"])


In [0]:
spark.read.table(full_table_name(dwh_schema,"fact_transactions")).show()

In [0]:
bal_source = incremental("daily_account_balance", "fact_account_balance").alias("o")
bal = (
    bal_source.join(
        accounts.alias("a"),
        (F.col("o.account_id") == F.col("a.AccountID"))
        & (F.to_date("o.balance_date") >= F.col("a.StartDate"))
        & (F.to_date("o.balance_date") < F.col("a.EndDate")),
    )
    .join(branches.alias("b"), F.col("a.BranchID") == F.col("b.BranchID"))
    .join(
        customers.alias("c"),
        (F.col("a.PrimaryCustomerID") == F.col("c.CustomerID"))
        & (F.to_date("o.balance_date") >= F.col("c.StartDate"))
        & (F.to_date("o.balance_date") < F.col("c.EndDate")),
    )
    .selectExpr( "CAST(date_format(o.balance_date,'yyyyMMdd') AS INT) DateSK", "o.balance_date BalanceDate", "o.account_id AccountID", "a.AccountSK", "c.CustomerSK", "b.BranchSK", "o.currency_code CurrencyCode", "o.available_balance_amount AvailableBalanceAmount", "o.ledger_balance_amount LedgerBalanceAmount", "o.overdraft_limit_amount OverdraftLimitAmount", "o.processing_date SourceIngestedAt",
    )
)
merge_df(bal, "fact_account_balance", ["DateSK", "AccountID"])

In [0]:
spark.read.table(full_table_name(dwh_schema,"fact_account_balance")).show()

In [0]:
lines = incremental("transaction_line", "bridge_transaction_product").alias('l')

lines = (lines.join(spark.table(full_table_name(dwh_schema, "dim_product")).select("ProductID", "ProductSK").alias('p'),(F.col("l.product_id") == F.col("p.ProductID")))
    .selectExpr("l.transaction_id TxnID","l.line_number LineNumber","ProductSK","l.product_id ProductID","l.quantity Quantity","l.line_amount LineAmount","processing_date SourceIngestedAt")
)
merge_df(lines, "bridge_transaction_product", ["TxnID", "LineNumber"])

In [0]:
spark.read.table(full_table_name(dwh_schema,"bridge_transaction_product")).show()

In [0]:
targets = ["dim_branch", "dim_product", "dim_account", "dim_customer", "bridge_account_customer", "fact_transactions", "fact_account_balance"]
watermark_rows = [(x, processing_date + " 23:59:59") for x in targets]
spark.createDataFrame(watermark_rows, ["target_table", "watermark_to"]).withColumn(
    "watermark_to", F.to_timestamp("watermark_to")
).withColumn("updated_at", F.current_timestamp()).write.mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(full_table_name(dwh_schema, "dwh_watermark"))

In [0]:

# Data-quality gates from the SQL notebook. Keep the result visible to the workflow run.
customer_duplicate_failures = (
    spark.table(full_table_name(dwh_schema, "dim_customer"))
    .filter(F.col("IsCurrent")).groupBy("CustomerID").count()
    .filter(F.col("count") != 1).count()
)
unresolved_customer_keys = (
    spark.table(full_table_name(dwh_schema, "fact_transactions"))
    .filter(F.col("CustomerSK").isNull())
    .count()
)
ownership_allocation_failures = (
    spark.table(full_table_name(dwh_schema, "bridge_account_customer"))
    .groupBy("AccountID", "StartDate").agg(F.abs(F.sum("OwnershipPercent") - F.lit(100)).alias("allocation_delta"))
    .filter(F.col("allocation_delta") > F.lit(0.01)).count()
)
dq_results = spark.createDataFrame([
    ("customer_current_duplicates", customer_duplicate_failures),
    ("unresolved_transaction_customer_keys", unresolved_customer_keys),
    ("ownership_allocation_not_100", ownership_allocation_failures)
    ], ["check_name", "failures"],
)
display(dq_results)